# Introduction au RAG (Retrieval Augmented Generation)

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/C-rBNv5ZbCn1Qe9a-c_RwQ.png" style="width:50%;margin:auto;display:flex" alt="indexing"/>

## Contexte

### Qu’est-ce que le RAG ?
L’une des applications les plus puissantes rendues possibles par les grands modèles de langage (LLM) est la création de chatbots sophistiqués de questions-réponses (Q&R). Il s’agit d’applications capables de répondre à des questions à partir d’informations sources spécifiques. Ces applications utilisent une technique appelée **génération augmentée par la recherche** (*retrieval-augmented generation*, RAG).  
Le RAG est une méthode permettant d’enrichir les connaissances d’un LLM avec des données supplémentaires, qui peuvent être vos propres données.

Les LLM peuvent raisonner sur un large éventail de sujets, mais leurs connaissances sont limitées aux données publiques disponibles jusqu’à la date de fin d’entraînement du modèle. Si vous souhaitez créer des applications d’IA capables de raisonner sur des données privées ou sur des données introduites après la date de coupure du modèle, vous devez enrichir les connaissances du modèle avec les informations spécifiques dont il a besoin. Le processus consistant à apporter et à insérer les informations appropriées dans le prompt du modèle est appelé **RAG**.

LangChain propose plusieurs composants conçus pour faciliter la création d’applications de questions-réponses et, plus généralement, d’applications basées sur le RAG.

### Architecture du RAG
Une application RAG typique se compose de deux éléments principaux :

* **Indexation** : Un pipeline permettant d’ingérer et d’indexer des données provenant d’une source. Cette étape se déroule généralement hors ligne.

* **Recherche et génération** : La chaîne RAG proprement dite prend la requête de l’utilisateur au moment de l’exécution, récupère les données pertinentes à partir de l’index, puis les transmet au modèle.

La séquence complète la plus courante, allant des données brutes à la réponse, ressemble aux exemples suivants.


- **Indexation**
1. **Chargement** : Tout d’abord, vous devez charger vos données. Cela se fait à l’aide des [DocumentLoaders](https://python.langchain.com/docs/how_to/#document-loaders).

2. **Découpage** : Les [séparateurs de texte (Text splitters)](https://python.langchain.com/docs/how_to/#text-splitters) divisent les `Documents` volumineux en segments plus petits. Cela est utile à la fois pour l’indexation des données et pour leur transmission au modèle, car les gros segments sont plus difficiles à rechercher et ne tiennent pas dans la fenêtre de contexte limitée d’un modèle.

3. **Stockage** : Vous avez besoin d’un endroit pour stocker et indexer ces segments afin qu’ils puissent être recherchés ultérieurement. Cela se fait généralement à l’aide d’un [VectorStore](https://python.langchain.com/docs/how_to/#vector-stores) et d’un modèle d’[Embeddings](https://python.langchain.com/docs/how_to/embed_text/).

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/WEE3pjeJvSZP0R7UL7CYTA.png" width="50%" alt="indexation"/> <br>
<span style="font-size: 10px;">[source](https://python.langchain.com/docs/tutorials/rag/)</span>

- **Recherche et génération**
1. **Recherche** : À partir de l’entrée de l’utilisateur, les segments pertinents sont récupérés depuis le stockage à l’aide d’un composant de recherche (*retriever*).
2. **Génération** : Un ChatModel / LLM produit une réponse en utilisant un prompt qui inclut la question et les données récupérées.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/SwPO26VeaC8VTZwtmWh5TQ.png" width="50%" alt="recherche"/> <br>
<span style="font-size: 10px;">[source](https://python.langchain.com/docs/use_cases/question_answering/)</span>


In [21]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

from langchain.docstore.document import Document as LangchainDocument
from typing import List

### Lecture du document

In [ ]:
filename = "Guide-Visiteur-CAN2021.pdf"
documents = PyPDFLoader(filename).load()

In [20]:
print(documents[20].page_content)

lE gUIDE DU vISITEUR
21
b) Cuisine Camerounaise / 
Cameroonian dishes
• Restaurant L ’Agora, chez Maïthé, 
rond-point Nlongkak Tél. 699 59 02 42 
• Chez Olivier 
• Kaba-Ngondo Centre-ville, face 
Direction des Impôts 
• Le Dauphin Bd de l’OUA près du 
Carrefour Coron  
• Le Municipal, B.P . 7 - Tél. : 
222 20 80 09 
• Le Rustique, B.P . 12300 - Tél. : 
222 23 20 00 
• Les Cigalons (Restaurant et 
Brasseries) Tél. : 222 23 41 25 
• Les Feuilles Vertes, quartier Elig-
Essono 
• Le Marseillais, Bd du Romain 
(centre-ville) (cantine, self-service) 
Centre commercial 
• Parallèle Club, Tél. 222 20 60 00 
• Restaurant du Lac (Hôtel des 
Députés) 
• Restaurant Les Lions du 
Cameroun, chez Mme OWONO 
Madeleine, Tél. 675 12 53 80 / 
675 02 49 21
• Delph Service, derrière usine 
Bastos Tél. 699 89 76 08 / 699 96 16 31 
• Restaurant Pili Pili, quartier 
Bastos, face ambassade d’Italie Tél. 
222 20 14 81 / 699 71 54 16 
• Restaurant Barracuda situé à 
l’ enceinte du manège, Carrefour 
Abbia; Tél. 

### Découpage du texte en morceau

In [22]:
text_splitter = RecursiveCharacterTextSplitter(
                        chunk_size=1000,
                        chunk_overlap=50,
                        add_start_index=True,
                        separators=["\n\n", "\n", ".", " ", ""],
                    )
    
chunks = text_splitter.split_documents(documents)
print(len(chunks))

156


In [44]:
type(chunks)

list

### Calcul des vecteurs d'embedding et stockage

In [26]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2", encode_kwargs={
            "normalize_embeddings": True
        })
docsearch = Chroma.from_documents(chunks, embeddings)
print("documents ingested")

documents ingested


### LLM

In [27]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
llm = ChatOpenAI(
    model=  "gpt-4.1-mini",
    temperature=0.5,
    #model_kwargs = {"top_p":0.2, "top_k":1},
    max_tokens=256,
    api_key=os.getenv("OPENAI_API_KEY")
)

Tous les briques sont prêtes, passons au quering.

In [36]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", 
    retriever=docsearch.as_retriever(), 
    return_source_documents=False
)

In [37]:
query = "Is Douala in cameroon ?"
qa.invoke(query)

{'query': 'Is Douala in cameroon ?',
 'result': 'Yes, Douala is in Cameroon. It is the economic capital of the country and a major city in the Littoral region.'}

In [38]:
prompt_template = """ Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

In [39]:
PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}

In [40]:
qa = RetrievalQA.from_chain_type(llm=llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)

query = "Where is Kribi ?"
qa.invoke(query)

{'query': 'Where is Kribi ?',
 'result': 'The document does not provide information about the location of Kribi.'}

In [41]:
qa = RetrievalQA.from_chain_type(llm=llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, 
                                 return_source_documents=False)

query = "Where is Douala ?"
qa.invoke(query)

{'query': 'Where is Douala ?',
 'result': 'Douala is located in the Littoral Region of Cameroon. It is situated near the Wouri River, which flows directly into the Atlantic Ocean. Douala serves as the main economic capital of Cameroon and is a strategic gateway to the countries of the CEMAC sub-region.'}

In [42]:
def touristic_agent():
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages= True)
    qa = ConversationalRetrievalChain.from_llm(
        llm=llm,
        chain_type="stuff",
        retriever=docsearch.as_retriever(),
        memory = memory,
        get_chat_history=lambda h : h, 
        return_source_documents=False
    )

    history = []
    while True:
        query = input("Question: ")
        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break

        result = qa({"question": query}, {"chat_history": history})
        history.append((query, result["answer"]))

        print("Answer", result["answer"])


In [43]:
touristic_agent()

Answer That's great! Douala is the economic capital of Cameroon and offers many interesting attractions and practical amenities for visitors. Here are some useful details for your visit:

- Douala is located in the Littoral Region and serves as a major gateway to the CEMAC sub-region.
- The city has a variety of hotels to suit different budgets. Some recommended hotels include:
  - Hôtel Sawa (4 stars) with 288 rooms and suites, located at 488 rue Verdun, Bonanjo. Prices start at about 69,000 FCFA.
  - K. Hotel (4 stars) on 729 rue Christian Tobie Kuoh, Bonanjo, with 114 rooms and apartments.
  - Hôtel Prince de Galles (3 stars) in Akwa, starting at 78,000 FCFA.
  - Planet Hotel (3 stars) on rue Boué Lapeyrère, Akwa, with 52 rooms and suites.
  - Hotel Bano Palace (3 stars) on Rue Drouot, facing Lycée d’ Akwa, with 100 rooms, apartments, and suites.
- For football fans, Douala has several prominent football academies such as Fundacio Samuel Eto'o Fils (FUNDESPORT), K
Answer Yaoundé is 